# TransFG — Stanford Cars (Aircraft 가중치 전이학습)

**논문**: TransFG: A Transformer Architecture for Fine-grained Recognition (AAAI 2022)  
**데이터셋**: Stanford Cars (자동차 196 모델 분류)

---

## 프로젝트 개요

이 노트북은 기존 `TransFG_Stanford_Cars.ipynb`와 동일한 학습을 수행하되,  
**초기 가중치를 Aircraft로 학습된 TransFG checkpoint**로 변경하여  
**가장 유사한 도메인 간 전이학습 효과**를 검증합니다.

### 왜 Aircraft 가중치인가?

| | CUB checkpoint | **Aircraft checkpoint** |
|---|---|---|
| 도메인 | 조류 (유기체) | **항공기 (인공물)** ← Cars와 가장 유사 |
| 텍스처 | 깃털·곡선 | **금속·직선** ← Cars와 동일 |
| 식별 단위 | 부리·날개·깃털 | **동체·날개·꼬리** ← Cars의 차체와 유사 |
| 모델 정확도 | 90.84% | 78.97% |

**가설**: Fine-grained 전이학습에서 **도메인 유사도가 절대 정확도보다 중요**하다면,  
Aircraft → Cars 전이가 CUB → Cars 전이보다 효과적일 것이다.

### 기존 노트북과의 차이

| 항목 | 기존 `TransFG_Stanford_Cars.ipynb` | **본 노트북 (Cars_New)** |
|------|-----------------------------------|--------------------------|
| 초기 가중치 | `pretrained/ViT-B_16.npz` (ImageNet-21k) | **Aircraft checkpoint (78.97%)** |
| 가중치 학습 도메인 | ImageNet-21k (일반 사물 14M장) | **FGVC-Aircraft (항공기 3.3K장, fine-grained)** |
| 분류 헤드 | 새로 초기화 (zero_head=True) | 새로 초기화 (Aircraft 100 → Cars 196, 클래스 수 다름) |
| 백본 (Embeddings, Transformer) | ImageNet-21k 가중치 | **Aircraft-trained TransFG 가중치** |
| 학습 steps | 200 (quick test) | 200 (quick test) — 동일 비교 |
| 그 외 모든 하이퍼파라미터 | — | 기존과 동일 |

### 비교 대상

기존 `TransFG_Stanford_Cars.ipynb` 200 steps Test Accuracy: **1.37%** (ImageNet-21k 초기화)

본 노트북 실행 후:
- **Aircraft > 1.37%** → 도메인 유사 전이학습 효과 입증
- **Aircraft 매우 큰 향상** → CUB 사용보다 Aircraft 사용이 유리하다는 가설 검증

### 도메인 비교 (Cars vs Aircraft)

| | Stanford Cars | FGVC-Aircraft |
|---|---|---|
| 대상 | 자동차 (인공물) | 항공기 (인공물) |
| 텍스처 | 금속·페인트 | 금속·페인트 |
| 차별화 포인트 | 차체 길이·헤드라이트·그릴 | 동체 길이·날개 후퇴각 |
| 클래스 | 196 모델 | 100 variant |
| Train | 8,144 | 3,334 |
| 논문 SOTA | 96.1% | 70.7% |

## 1. 환경 확인

In [ ]:
import sys
import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. 모듈 임포트 및 경로 설정

Aircraft checkpoint를 초기 가중치로 사용합니다.

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# TransFG 내부 모듈
TRANSFG_ROOT = Path("TransFG").resolve()
if str(TRANSFG_ROOT) not in sys.path:
    sys.path.insert(0, str(TRANSFG_ROOT))

from models.modeling import VisionTransformer, CONFIGS

# Stanford Cars 전용 모듈
from dataset_stanford_cars import StanfordCarsDataset
from data_loader_cars      import get_cars_loaders

# 공통 유틸
from trainer         import train, validate
from inference_utils import predict_batch, evaluate_dataset
from visualization   import show_sample_grid, plot_history, visualize_predictions, visualize_attention

DEVICE              = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT           = Path("data/Stanford_Cars")
AIRCRAFT_CHECKPOINT = Path("output/fgvc_aircraft_from_cub/transfg_aircraft_from_cub_checkpoint.bin")  # ← Aircraft 가중치
OUTPUT_DIR          = Path("output/stanford_cars_from_aircraft")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device              : {DEVICE}")
print(f"DATA_ROOT           : {DATA_ROOT.resolve()}")
print(f"AIRCRAFT_CHECKPOINT : {AIRCRAFT_CHECKPOINT.resolve()}")
print(f"OUTPUT_DIR          : {OUTPUT_DIR.resolve()}")

assert DATA_ROOT.exists(),           "데이터셋 없음 — data/Stanford_Cars 확인"
assert AIRCRAFT_CHECKPOINT.exists(), f"Aircraft 가중치 없음 — {AIRCRAFT_CHECKPOINT}"
print(f"\nAircraft 가중치 크기: {AIRCRAFT_CHECKPOINT.stat().st_size/1e6:.0f} MB")

# 한글 폰트 설정
import matplotlib.font_manager as _fm
_font = Path("NotoSansCJK-Regular.ttc")
if _font.exists():
    _fm.fontManager.addfont(str(_font.resolve()))
    plt.rcParams["font.family"] = "Noto Sans CJK JP"

## 3. 데이터셋 탐색

In [ ]:
trainset_raw = StanfordCarsDataset(root=str(DATA_ROOT), split="train", transform=None)
testset_raw  = StanfordCarsDataset(root=str(DATA_ROOT), split="test",  transform=None)

print(f"Train : {len(trainset_raw):>5}장")
print(f"Test  : {len(testset_raw):>5}장")
n_classes = len(set(trainset_raw.labels))
print(f"클래스: {n_classes}종")

## 4. 하이퍼파라미터 설정

**기존 Cars 노트북과 100% 동일** (공정한 비교를 위해).

In [ ]:
CFG = {
    "model_type"  : "ViT-B_16",
    "split"       : "overlap",
    "slide_step"  : 12,
    "img_size"    : 448,
    "num_classes" : 196,                            # Stanford Cars: 196종 (Aircraft는 100)
    "train_batch" : 8,
    "eval_batch"  : 8,
    "num_steps"   : 10000,
    "quick_steps" : 200,
    "lr"          : 3e-2,
    "warmup_steps": 500,
    "decay_type"  : "cosine",
    "smoothing"   : 0.0,
    "fp16"        : True,
    "run_name"    : "transfg_cars_from_aircraft",   # Aircraft 가중치 사용 표시
    "num_workers" : 16,
}

print("=== TransFG Stanford Cars (Aircraft init) Config ===")
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

## 5. DataLoader 생성

In [ ]:
train_loader, test_loader, trainset, testset = get_cars_loaders(
    data_root        = str(DATA_ROOT),
    train_batch_size = CFG["train_batch"],
    eval_batch_size  = CFG["eval_batch"],
    img_size         = CFG["img_size"],
    num_workers      = CFG["num_workers"],
)

x_batch, y_batch = next(iter(train_loader))
print(f"Input batch  : {x_batch.shape}")
print(f"Label batch  : {y_batch.shape}")
print(f"Train 배치 수: {len(train_loader)}")
print(f"Test  배치 수: {len(test_loader)}")

## 6. 모델 생성 및 Aircraft 가중치 로드 ⭐

**이 노트북의 핵심 차이점.**

### 가중치 로드 전략

1. **모델 생성**: `num_classes=196` (Cars) 으로 빈 모델 생성
2. **Aircraft checkpoint 로드**: `part_head` 가중치 제외 (Aircraft 100 vs Cars 196 크기 불일치)
3. **분류 헤드**: PyTorch 기본 초기화 (랜덤) 후 학습

Aircraft checkpoint는 CUB 학습 가중치로 시작해 Aircraft로 fine-tune된 모델.  
즉 **CUB → Aircraft → Cars** 3단계 전이학습.

In [ ]:
# Step 1: 모델 생성 (Cars 196 classes)
config = CONFIGS[CFG["model_type"]]
config.split      = CFG["split"]
config.slide_step = CFG["slide_step"]

p_size = config.patches["size"][0]
n_side = (CFG["img_size"] - p_size) // CFG["slide_step"] + 1
print(f"patch grid : {n_side}×{n_side} = {n_side**2} patches\n")

model = VisionTransformer(
    config,
    img_size        = CFG["img_size"],
    zero_head       = True,
    num_classes     = CFG["num_classes"],   # 196 (Cars)
    smoothing_value = CFG["smoothing"],
)

# Step 2: Aircraft checkpoint 로드 (head 가중치 제외)
print(f"Aircraft checkpoint 로드 중: {AIRCRAFT_CHECKPOINT}")
ckpt = torch.load(str(AIRCRAFT_CHECKPOINT), map_location="cpu")
aircraft_state_dict = ckpt["model"]

# checkpoint 정보 출력
print(f"  - Aircraft 가중치 키 개수: {len(aircraft_state_dict)}")
print(f"  - Aircraft 학습 step      : {ckpt.get('global_step', 'N/A')}")
print(f"  - Aircraft 최고 정확도    : {ckpt.get('best_acc', 0.0):.4f}")

# 크기 불일치 layer 제거 (part_head: 100 → 196)
filtered_state_dict = {
    k: v for k, v in aircraft_state_dict.items()
    if not k.startswith("part_head")
}
removed_keys = set(aircraft_state_dict.keys()) - set(filtered_state_dict.keys())
print(f"\n  - head 가중치 제외 (Aircraft 100 vs Cars 196):")
for k in removed_keys:
    print(f"      · {k}  (제외됨)")

missing, unexpected = model.load_state_dict(filtered_state_dict, strict=False)

print(f"\n  - missing : {len(missing)}")
for k in missing:
    print(f"      · {k}")
print(f"  - unexpected: {len(unexpected)}")
for k in unexpected:
    print(f"      · {k}")

model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n학습 파라미터: {n_params/1e6:.1f}M")
print(f"분류 헤드    : {model.part_head}  ← 새로 초기화됨")

model.eval()
with torch.no_grad():
    out = model(torch.randn(2, 3, CFG["img_size"], CFG["img_size"]).to(DEVICE))
print(f"추론 출력    : {out.shape}  (배치=2, 클래스=196)")

## 7. 빠른 학습 테스트 (200 steps)

기존 노트북과 **동일 설정**으로 200 steps 학습.  
결과를 기존 ImageNet-21k 베이스라인 (test_acc=1.37%) 과 비교합니다.

In [ ]:
import time

print(f"Quick training: {CFG['quick_steps']} steps  |  init: Aircraft checkpoint")
print("=" * 70)

start = time.time()
best_acc, history = train(
    model        = model,
    train_loader = train_loader,
    test_loader  = test_loader,
    device       = DEVICE,
    num_steps    = CFG["quick_steps"],
    learning_rate= CFG["lr"],
    warmup_steps = 50,
    decay_type   = CFG["decay_type"],
    eval_every   = 50,
    output_dir   = str(OUTPUT_DIR),
    run_name     = CFG["run_name"],
    fp16         = CFG["fp16"],
    gradient_accumulation_steps = 1,
    max_grad_norm = 1.0,
)
elapsed = time.time() - start
print(f"\n경과 시간    : {elapsed:.1f}초 ({elapsed/60:.1f}분)")
print(f"Best val_acc : {best_acc:.4f} ({best_acc*100:.2f}%)")

## 8. 학습 곡선 시각화

In [ ]:
plot_history(history, eval_every=50)

## 9. ImageNet-21k 베이스라인과 비교

동일한 200 steps 학습 결과를 비교.

In [ ]:
# 기존 ImageNet-21k 베이스라인 (TransFG_Stanford_Cars.ipynb 결과)
# 200 steps 후 Test Accuracy: 1.37%
baseline_imagenet21k_final = 0.0137

# 본 노트북 결과 (Aircraft checkpoint)
aircraft_init_results = {}
for i, acc in enumerate(history["val_acc"]):
    step = (i + 1) * 50
    aircraft_init_results[step] = acc

# 비교 출력
print("=" * 60)
print("200 steps 결과 비교")
print("=" * 60)
print(f"{'초기화':<25}: {'Test/Val Acc':>12}")
print("-" * 60)
print(f"{'ImageNet-21k (baseline)':<25}: {baseline_imagenet21k_final*100:>10.2f}%")
for step, acc in sorted(aircraft_init_results.items()):
    label = f"Aircraft checkpoint (step {step})"
    print(f"{label:<25}: {acc*100:>10.2f}%")
print("=" * 60)

# 결론
final_acc = aircraft_init_results.get(200, 0.0)
diff = final_acc - baseline_imagenet21k_final
print(f"\n[200 steps 차이]")
print(f"  Aircraft init: {final_acc*100:.2f}%")
print(f"  ImageNet-21k : {baseline_imagenet21k_final*100:.2f}%")
print(f"  차이         : {diff*100:+.2f}%p")

if diff > 0.05:
    print(f"\n  → 도메인 유사 전이학습 효과 강력하게 확인")
elif diff > 0.0:
    print(f"\n  → 전이학습 효과 일부 확인")
else:
    print(f"\n  → 효과 미미 또는 음의 효과")

## 10. 전체 학습 (Optional, 10,000 steps)

In [ ]:
# 전체 학습 — 주석 해제 후 실행
# import time
#
# # 모델 재생성 + Aircraft 가중치 재로드 (head 제외)
# model = VisionTransformer(config, img_size=CFG["img_size"],
#                           zero_head=True, num_classes=CFG["num_classes"],
#                           smoothing_value=CFG["smoothing"])
# ckpt = torch.load(str(AIRCRAFT_CHECKPOINT), map_location="cpu")
# filtered_state_dict = {k: v for k, v in ckpt["model"].items() if not k.startswith("part_head")}
# model.load_state_dict(filtered_state_dict, strict=False)
# model = model.to(DEVICE)
#
# start = time.time()
# best_acc, history_full = train(
#     model        = model,
#     train_loader = train_loader,
#     test_loader  = test_loader,
#     device       = DEVICE,
#     num_steps    = CFG["num_steps"],          # 10,000
#     learning_rate= CFG["lr"],
#     warmup_steps = CFG["warmup_steps"],       # 500
#     decay_type   = CFG["decay_type"],
#     eval_every   = 100,
#     output_dir   = str(OUTPUT_DIR),
#     run_name     = CFG["run_name"],
#     fp16         = CFG["fp16"],
#     gradient_accumulation_steps = 1,
#     max_grad_norm = 1.0,
# )
# print(f"\n경과 시간: {(time.time()-start)/3600:.2f}시간")
# print(f"Best val_acc: {best_acc:.4f}")

## 요약

| 셀 | 내용 |
|---|---|
| 0 | 프로젝트 개요 — Aircraft 가중치 선택 이유 |
| 1~4 | 환경 확인, 모듈 임포트 (Aircraft checkpoint 추가) |
| 5~6 | 데이터셋 탐색 |
| 7~8 | 하이퍼파라미터 설정 (기존과 동일) |
| 9~10 | DataLoader 생성 |
| 11~12 | **모델 생성 + Aircraft 가중치 로드** ⭐ |
| 13~14 | 빠른 학습 테스트 (200 steps) |
| 15~16 | 학습 곡선 시각화 |
| 17~18 | **ImageNet-21k 베이스라인과 비교** ⭐ |
| 19~20 | 전체 학습 (Optional, 10k steps) |